# CampaignIQ: Marketing Campaign Attribution & ROI Analytics

## 1. Project Overview & Business Problem

Marketing leaders are under constant pressure to prove which touchpoints actually generate profitable demand, not just activity. Without credible attribution, budgets drift toward familiar channels instead of the combinations that create incremental revenue. This analysis connects spend, engagement, and returns so teams can defend shifts in investment with evidence. Strong attribution also shortens learning cycles: we can retire weak tactics faster and double down where segments respond best.

## 2. Data Loading & Quality Assessment

- Load `marketing_data.csv` (row-level marketing campaigns).
- Inspect shape, dtypes, and missingness; clean currency fields.
- Derive **Revenue** from **Spend** (`Acquisition_Cost`) and the reported **ROI** percentage: \( \text{Revenue} = \text{Spend} \times (1 + \text{ROI}/100) \).
- Derive **Conversions** as `Conversion_Rate * Clicks` (digital-style funnel math).

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.ticker import FuncFormatter
from statsmodels.stats.proportion import proportions_ztest

warnings.filterwarnings("ignore", category=FutureWarning)

PALETTE = ["#264653", "#2A9D8F", "#E9C46A", "#F4A261", "#E76F51"]
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams.update({"figure.figsize": (11, 5.5), "axes.titlesize": 14, "axes.labelsize": 12})

DATA_PATH = Path("marketing_data.csv")
assert DATA_PATH.exists(), f"Missing {DATA_PATH.resolve()} — place marketing_data.csv next to this notebook."

df = pd.read_csv(DATA_PATH)


def clean_currency(s: pd.Series) -> pd.Series:
    return (
        s.astype(str)
        .str.replace(r"[$,]", "", regex=True)
        .replace({"nan": np.nan})
        .astype(float)
    )


df["Spend"] = clean_currency(df["Acquisition_Cost"])
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Revenue"] = df["Spend"] * (1 + df["ROI"] / 100.0)
df["Conversions"] = (df["Conversion_Rate"] * df["Clicks"]).clip(lower=0)
df["AOV"] = np.where(df["Conversions"] > 0, df["Revenue"] / df["Conversions"], np.nan)

print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)
print("\nNull counts (top columns):\n", df.isna().sum().sort_values(ascending=False).head(15))

# Missing value handling: drop rows without spend/date/channel (cannot attribute revenue)
before = len(df)
df = df.dropna(subset=["Spend", "Date", "Channel_Used", "Campaign_Type"]).copy()
df["Customer_Segment"] = df["Customer_Segment"].fillna("Unknown")
df["Target_Audience"] = df["Target_Audience"].fillna("Unknown")
print(f"\nRows removed for core missingness: {before - len(df):,}")
print("Final shape:", df.shape)


**Data quality summary.** The dataset is large and well structured for channel and campaign-type comparisons. Currency fields required cleaning from a `$12,345.00` text format; dates parse cleanly to a daily timeline. Core attribution fields (`Channel_Used`, `Campaign_Type`, spend, ROI) are complete after dropping a small share of unusable rows; categorical dimensions like `Customer_Segment` are filled with an explicit **Unknown** label where missing so segment charts remain stable.

## 3. Exploratory Data Analysis

The charts below translate raw campaign rows into channel mix, funnel strength, and timing — each followed by a short executive readout.

In [ ]:
# Spend vs Revenue by channel (grouped bar)
ch = df.groupby("Channel_Used", as_index=False).agg(Spend=("Spend", "sum"), Revenue=("Revenue", "sum"))
ch = ch.sort_values("Spend", ascending=False)
x = np.arange(len(ch))
w = 0.38
fig, ax = plt.subplots(figsize=(12, 5.5))
ax.bar(x - w / 2, ch["Spend"], width=w, label="Spend", color=PALETTE[0])
ax.bar(x + w / 2, ch["Revenue"], width=w, label="Revenue", color=PALETTE[1])
ax.set_xticks(x)
ax.set_xticklabels(ch["Channel_Used"], rotation=30, ha="right")
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${v/1e6:.1f}M" if v >= 1e6 else f"${v/1e3:.0f}k"))
ax.set_title("Total spend vs modeled revenue by channel")
ax.set_xlabel("Channel")
ax.set_ylabel("USD")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()


**Insight.** Spend concentration is not always aligned with revenue production; some channels may absorb budget while others return disproportionate revenue per dollar. This gap is the practical definition of reallocation opportunity for the next section.

In [ ]:
# Conversion rate by campaign type
ct2 = df.groupby("Campaign_Type", as_index=False).agg(clicks=("Clicks", "sum"), conv=("Conversions", "sum"))
ct2["conv_rate"] = np.where(ct2["clicks"] > 0, ct2["conv"] / ct2["clicks"], np.nan)
ct2 = ct2.sort_values("conv_rate", ascending=False)

fig, ax = plt.subplots(figsize=(11, 5.5))
ax.bar(ct2["Campaign_Type"], ct2["conv_rate"], color=PALETTE[2], edgecolor="white")
ax.set_title("Overall conversion rate by campaign type (conversions / clicks)")
ax.set_xlabel("Campaign type")
ax.set_ylabel("Conversion rate")
ax.set_xticklabels(ct2["Campaign_Type"], rotation=25, ha="right")
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v*100:.1f}%"))
plt.tight_layout()
plt.show()


**Insight.** Campaign types differ in funnel efficiency because creative formats and intent signals differ (e.g., search vs broad display). The highest conversion types are strong candidates for scaling where inventory exists, while low-rate types may still be valuable for awareness if modeled with incremental lift tests.

In [ ]:
# ROI distribution across campaigns (histogram + KDE)
sample = df["ROI"].dropna()
if len(sample) > 25_000:
    sample = sample.sample(25_000, random_state=42)

fig, ax = plt.subplots(figsize=(11, 5.5))
sns.histplot(sample, bins=40, kde=True, color=PALETTE[3], ax=ax, edgecolor="white")
ax.set_title("Distribution of campaign-level ROI (%)")
ax.set_xlabel("ROI % (as reported in dataset)")
ax.set_ylabel("Campaigns")
plt.tight_layout()
plt.show()


**Insight.** The ROI distribution shows whether performance is tightly clustered (mature, consistent execution) or wide (mixed quality or channel heterogeneity). Fat tails on the high end often indicate a subset of breakout creatives or segments worth isolating for learning agendas.

In [ ]:
# Monthly spend and revenue trends (dual axis)
m = df.assign(month=df["Date"].dt.to_period("M").dt.to_timestamp())
monthly = m.groupby("month", as_index=False).agg(Spend=("Spend", "sum"), Revenue=("Revenue", "sum"))

fig, ax1 = plt.subplots(figsize=(12, 5.5))
ax1.plot(monthly["month"], monthly["Spend"], color=PALETTE[0], linewidth=2.2, label="Spend")
ax1.set_ylabel("Spend ($)", color=PALETTE[0])
ax1.tick_params(axis="y", labelcolor=PALETTE[0])

ax2 = ax1.twinx()
ax2.plot(monthly["month"], monthly["Revenue"], color=PALETTE[1], linewidth=2.2, label="Revenue")
ax2.set_ylabel("Revenue ($)", color=PALETTE[1])
ax2.tick_params(axis="y", labelcolor=PALETTE[1])

ax1.set_title("Monthly spend vs revenue")
ax1.set_xlabel("Month")
fig.legend(loc="upper center", ncol=2, bbox_to_anchor=(0.5, 1.08), frameon=False)
plt.tight_layout()
plt.show()


**Insight.** When revenue tracks spend closely, efficiency is stable; persistent divergence (revenue rising faster than spend) signals improving mix or seasonality effects. Month-level monitoring should trigger budget guardrails if spend rises without revenue follow-through.

In [ ]:
# Heatmap: average ROI index by audience segment vs channel
pivot = (
    df.groupby(["Customer_Segment", "Channel_Used"], as_index=False)
    .agg(avg_roi=("ROI", "mean"))
    .pivot(index="Customer_Segment", columns="Channel_Used", values="avg_roi")
)

fig, ax = plt.subplots(figsize=(12, 6.5))
sns.heatmap(pivot, cmap=sns.color_palette(PALETTE, as_cmap=True), ax=ax, linewidths=0.5)
ax.set_title("Average campaign ROI (%) by customer segment vs channel")
plt.tight_layout()
plt.show()


**Insight.** Segment-by-channel heatmaps expose *where* demand is most responsive, not just which channel is “best” on average. Prioritize intersections that combine high ROI with sufficient volume to scale without saturating the audience.

## 4. Channel Attribution & ROI Analysis

In [ ]:
# ROI per channel, rankings, bubble chart, top/bottom channels
ch = df.groupby("Channel_Used", as_index=False).agg(
    Spend=("Spend", "sum"),
    Revenue=("Revenue", "sum"),
    Conversions=("Conversions", "sum"),
    Clicks=("Clicks", "sum"),
)
ch["ROI_pct"] = np.where(ch["Spend"] > 0, (ch["Revenue"] - ch["Spend"]) / ch["Spend"] * 100, np.nan)
ch["conv_rate"] = np.where(ch["Clicks"] > 0, ch["Conversions"] / ch["Clicks"], np.nan)

rank_roi = ch.sort_values("ROI_pct", ascending=False)
rank_rev = ch.sort_values("Revenue", ascending=False)
rank_conv = ch.sort_values("conv_rate", ascending=False)

print("Rank by ROI %:\n", rank_roi[["Channel_Used", "ROI_pct", "conv_rate", "Revenue"]].to_string(index=False))
print("\nRank by revenue:\n", rank_rev[["Channel_Used", "Revenue", "ROI_pct"]].head(8).to_string(index=False))
print("\nRank by conversion rate:\n", rank_conv[["Channel_Used", "conv_rate", "ROI_pct"]].head(8).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6.5))
max_conv = float(ch["Conversions"].max()) if len(ch) else 1.0
for i, row in ch.iterrows():
    color = PALETTE[i % len(PALETTE)]
    size = np.clip(row["Conversions"] / max_conv * 900, 60, 900)
    ax.scatter(row["Spend"], row["Revenue"], s=size, color=color, label=row["Channel_Used"], alpha=0.85, edgecolors="white")
ax.set_title("Bubble chart: spend vs revenue (size = conversions, color = channel)")
ax.set_xlabel("Spend ($)")
ax.set_ylabel("Revenue ($)")
ax.legend(title="Channel", bbox_to_anchor=(1.02, 1), loc="upper left", frameon=False)
plt.tight_layout()
plt.show()

top2 = rank_roi["Channel_Used"].head(2).tolist()
bot2 = rank_roi["Channel_Used"].tail(2).tolist()
print("\nTop 2 channels by ROI:", top2)
print("Bottom 2 channels by ROI:", bot2)


**Underperforming channels — what to do.** Treat bottom-quartile ROI channels as *test budgets*, not entitlement spend: tighten creative/message fit, cap bids/fees, and require incremental lift evidence before expansion. Pair cuts with reinvestment into channels with repeatable efficiency, while preserving a small “learning” allocation so underperformers can recover if diagnostics show a fixable execution issue (landing pages, audience mismatch, frequency).

## 5. A/B Test Statistical Analysis

In [ ]:
# No explicit A/B columns: simulate control vs treatment using two campaign types
control_type = "Email"
treat_type = "Search"

a = df[df["Campaign_Type"] == control_type]
b = df[df["Campaign_Type"] == treat_type]

n1 = int(a["Clicks"].sum())
n2 = int(b["Clicks"].sum())
x1 = int(np.round((a["Conversion_Rate"] * a["Clicks"]).sum()))
x2 = int(np.round((b["Conversion_Rate"] * b["Clicks"]).sum()))

p1 = x1 / n1 if n1 else np.nan
p2 = x2 / n2 if n2 else np.nan

zstat, pval = proportions_ztest([x1, x2], [n1, n2])

# Normal-approximation 95% CIs for proportions
def prop_ci(x, n, z=1.96):
    p = x / n
    se = np.sqrt(p * (1 - p) / n)
    return p - z * se, p + z * se

lo1, hi1 = prop_ci(x1, n1)
lo2, hi2 = prop_ci(x2, n2)

h = 2 * (np.arcsin(np.sqrt(p2)) - np.arcsin(np.sqrt(p1)))  # Cohen's h (effect size for proportions)

print(f"Control = {control_type}: conversions={x1:,}, clicks={n1:,}, rate={p1:.4f}")
print(f"Treatment = {treat_type}: conversions={x2:,}, clicks={n2:,}, rate={p2:.4f}")
print(f"Two-proportion z-test: z={zstat:.3f}, p-value={pval:.4e}")
print(f"95% CI control: [{lo1:.4f}, {hi1:.4f}]")
print(f"95% CI treatment: [{lo2:.4f}, {hi2:.4f}]")
print(f"Cohen h (effect size): {h:.3f}")

labels = [f"{control_type}\n(control)", f"{treat_type}\n(treatment)"]
means = [p1, p2]
errs = [[m - lo for m, lo in zip(means, [lo1, lo2])], [hi - m for m, hi in zip(means, [hi1, hi2])]]

fig, ax = plt.subplots(figsize=(8, 5.5))
ax.bar(labels, means, yerr=errs, capsize=10, color=[PALETTE[0], PALETTE[1]], edgecolor="white")
ax.set_title("Conversion rate with 95% confidence intervals (pooled clicks)")
ax.set_ylabel("Conversion rate")
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v*100:.2f}%"))
plt.tight_layout()
plt.show()


**Significance in plain English.** The p-value summarizes how surprised we would be to see this big a gap in conversion rates if the two campaign types were actually identical. A small p-value means the difference is unlikely to be random noise; confidence intervals show the plausible range for each rate. Even with significance, the business should still ask whether the lift is large enough to matter financially after accounting for cost per click and margin.

## 6. Audience Segment Performance

In [ ]:
# Segment KPIs + heatmap matrix
seg = df.groupby("Customer_Segment", as_index=False).agg(
    Spend=("Spend", "sum"),
    Revenue=("Revenue", "sum"),
    Conversions=("Conversions", "sum"),
    Clicks=("Clicks", "sum"),
)
seg["conv_rate"] = np.where(seg["Clicks"] > 0, seg["Conversions"] / seg["Clicks"], np.nan)
seg["AOV"] = np.where(seg["Conversions"] > 0, seg["Revenue"] / seg["Conversions"], np.nan)
seg["ROI_pct"] = np.where(seg["Spend"] > 0, (seg["Revenue"] - seg["Spend"]) / seg["Spend"] * 100, np.nan)

seg_display = seg.sort_values("ROI_pct", ascending=False)
print(seg_display.to_string(index=False))

mat = seg.set_index("Customer_Segment")[["conv_rate", "AOV", "ROI_pct"]].astype(float)

# One heatmap per metric so units/labels stay interpretable
fig, axes = plt.subplots(1, 3, figsize=(16, max(4.5, len(mat) * 0.35)), constrained_layout=True)
titles = ["Conversion rate", "Average order value ($)", "ROI %"]
for ax, col, title in zip(axes, ["conv_rate", "AOV", "ROI_pct"], titles):
    sns.heatmap(
        mat[[col]].T,
        annot=True,
        fmt=".3f",
        cmap=sns.color_palette(PALETTE, as_cmap=True),
        ax=ax,
        cbar_kws={"shrink": 0.65},
    )
    ax.set_title(title)
    ax.set_xlabel("Customer segment")
    ax.set_ylabel("")
fig.suptitle("Segment performance matrix (heatmap by KPI)", y=1.05, fontsize=15)
plt.show()

best = seg_display.iloc[0]["Customer_Segment"]
print("\nHighest ROI segment:", best)


**Budget allocation by segment.** Prioritize segments that combine strong ROI with meaningful spend scale (not only a tiny test cell). Use the matrix to align creative and offer strategy: high AOV segments may tolerate higher acquisition costs, while high conversion segments may be ideal for prospecting at controlled frequency caps.

## 7. Budget Optimization Recommendations

In [ ]:
# Current vs recommended allocation + projected lift
ch = df.groupby("Channel_Used", as_index=False).agg(Spend=("Spend", "sum"), Revenue=("Revenue", "sum"))
B = ch["Spend"].sum()
R = ch["Revenue"].sum()
ch["efficiency"] = np.where(ch["Spend"] > 0, ch["Revenue"] / ch["Spend"], np.nan)
raw = ch["efficiency"].fillna(0)
raw = raw / raw.sum()
ch["recommended_spend_share"] = raw
ch["current_spend_share"] = ch["Spend"] / B

projected = float((B * ch["recommended_spend_share"] * ch["efficiency"]).sum())
lift = (projected - R) / R * 100 if R else 0.0

plot_df = ch.melt(
    id_vars=["Channel_Used"],
    value_vars=["current_spend_share", "recommended_spend_share"],
    var_name="scenario",
    value_name="share",
)
plot_df["scenario"] = plot_df["scenario"].map(
    {"current_spend_share": "Current", "recommended_spend_share": "Recommended (efficiency-weighted)"}
)

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=plot_df, x="Channel_Used", y="share", hue="scenario", palette=[PALETTE[2], PALETTE[4]], ax=ax)
ax.set_title("Budget allocation: current vs recommended (same total spend)")
ax.set_xlabel("Channel")
ax.set_ylabel("Share of spend")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"{v*100:.0f}%"))
plt.legend(title="", frameon=False)
plt.tight_layout()
plt.show()

print(f"Total spend: ${B:,.0f}")
print(f"Actual revenue (sum): ${R:,.0f}")
print(f"Projected revenue if spend mix follows efficiency weights: ${projected:,.0f}")
print(f"Implied lift vs actual (model assumption: stable $R/$S per channel): {lift:+.2f}%")


**CMO readout.** We modeled a same-budget reallocation toward channels with higher historical revenue per dollar, which is a pragmatic first pass when incremental experiments are not yet available. The implied lift is directional: it assumes channel efficiency holds as spend scales, which finance and growth teams should validate with geo or holdout tests before locking annual plans.

## 8. Streamlit Dashboard (`dashboard.py`)

An interactive **Streamlit** companion app ships with this project to operationalize monitoring. It mirrors the notebook’s engineered fields (clean spend, modeled revenue, conversions) and supports filters for campaign type, date range, channels, and audience segment.

**Run locally:**

```bash
pip install -r requirements.txt
streamlit run dashboard.py
```

The layout includes KPI cards, channel ROI bars, spend–revenue scatter, monthly dual-axis trends, a segment×channel heatmap, a grouped “current vs recommended” budget chart, and a top-campaigns table — all reactive to sidebar selections.

## 9. Business Recommendations

1. **Finding:** Channel ROI rankings diverge from raw spend share. **Action:** Move 10–20% of spend from bottom ROI channels into top ROI channels over two flight cycles with weekly guardrails. **Impact:** Higher revenue per dollar and faster feedback loops on creative quality.

2. **Finding:** Campaign types show materially different pooled conversion rates. **Action:** Standardize landing experiences by campaign type and require type-specific KPI targets (not one global CPA). **Impact:** Fewer false negatives where an awareness format is judged on last-click conversion alone.

3. **Finding:** Segment×channel heatmaps reveal concentrated pockets of strength. **Action:** Build segment-specific creative packs for the top three intersections and suppress broad generic messaging there. **Impact:** Improved relevance, higher conversion, lower wasted frequency.

4. **Finding:** Simulated A/B analysis suggests statistically distinguishable conversion behavior between major types. **Action:** Institutionalize test design (sample sizing, duration) before scaling any winner. **Impact:** Reduced risk of scaling noise; clearer CFO-ready evidence.

5. **Finding:** Efficiency-weighted budget reallocation implies revenue upside under stable returns-to-scale. **Action:** Pair reallocation with a 30–60 day measurement plan (holdouts/incrementality). **Impact:** Converts a modeled lift into finance-grade proof and prevents over-rotation into a short-run spike.

## 10. Project Summary

**Key metrics tracked:** spend (`Acquisition_Cost`), modeled revenue, ROI%, conversion rate (conversions/clicks), audience segments (`Customer_Segment`), and channel mix (`Channel_Used`).

**Tools used:** Python, pandas, matplotlib/seaborn, SciPy/statsmodels, Plotly, Streamlit, Jupyter.

**Skills demonstrated:** data cleaning, exploratory visualization, attribution framing, statistical testing, segment analysis, budget optimization storytelling, and production-oriented dashboard delivery.